# Task 4 — Controlled leaf-classifier experiments

**Prepared notebook; no training results are included.** Open this file in Google Colab,
select **Runtime → Change runtime type → T4 GPU**, enter your teacher's original group
ZIP file URL below, and choose **Runtime → Run all**. Allow Google Drive mounting when
asked. No other code edits are needed. The ZIP must be shared as “Anyone with the link — Viewer”.

Default: **B0, E1, E5, E6, E7**, each with seeds **0, 1, 2** and **15 epochs**.
These priority runs finish before optional experiments. `RUN_ALL` additionally enables
E2, E3, E4 and E8; conditional E9 runs only when completed repeated comparisons support
combining improvements. Leave `RUN_ALL` off for the shorter first run.

All results, per-epoch checkpoints, generated validation photographs and pretrained weights
stay in your **private Google Drive / MyDrive / MLR_ex08_Task4**. After a disconnect,
select the same GPU runtime and run all again with the same URL: completed runs are skipped,
and interrupted runs resume at the last saved epoch. At most the unsaved epoch must be repeated.
Keep enough Drive space (approximately 0.5 GB for the priority runs).

This copies `LeafDataset` and `LeafNet` from your completed Task 3 notebook. The original
notebook, deployed website and original `leaf_model.onnx` are never edited. Neither
`REPORT.md` nor `REVIEW.md` is written. Do not publish an executed notebook containing
your group URL, generated photographs or private checkpoint folders.

Official instructions: [course assignment](https://tomasvicar.github.io/MLR-public/exercises/ex08.html),
verified against the [official course notebook](https://github.com/tomasvicar/MLR-public/blob/master/exercises/ex08/ex08_student.ipynb)
(Git blob `3b07efc118aeefb8b342fd4f6c2dc0094cea5115`).

In [ ]:
GROUP_ZIP = ""  # @param {type:"string"}
RUN_ALL = False  # @param {type:"boolean"}

SEEDS = [0, 1, 2]
EPOCHS = 15
BATCH_SIZE = 32
LR = 1e-3
CAMERA_SEED = 20261111
EXPECTED_DATASET_SHA256 = "880787da82a68668ce3f650e7094b06b509869f9ef71e86dc84cf57e5f6de253"

## 1. Colab environment and persistent storage

Colab provides PyTorch, torchvision, NumPy, pandas, Pillow and matplotlib. Only missing
download/export dependencies are installed. An environment/configuration fingerprint separates
incompatible runs, so new package versions cannot silently resume older checkpoints.
Fixed seeds and deterministic cuDNN settings improve repeatability within this environment;
bitwise equality across different GPU hardware or PyTorch releases is not promised.

In [ ]:
import importlib.util
import subprocess
import sys

missing = [name for name in ["gdown", "onnx", "onnxruntime"] if importlib.util.find_spec(name) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

import hashlib
import json
import os
import random
import time
import traceback
import zipfile
from pathlib import Path
from urllib.parse import parse_qs, urlparse

os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
import gdown
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import PIL
import torch
import torch.nn as nn
import torchvision
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from google.colab import drive, files

assert torch.cuda.is_available(), "Select a T4 GPU runtime, then run all again."
device = torch.device("cuda")
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
drive.mount("/content/drive")
ROOT = Path("/content/drive/MyDrive/MLR_ex08_Task4")
ROOT.mkdir(parents=True, exist_ok=True)
torch.hub.set_dir(str(ROOT / "torch_cache"))
environment = {"python": sys.version.split()[0], "torch": torch.__version__,
               "torchvision": torchvision.__version__, "numpy": np.__version__,
               "pillow": PIL.__version__, "cuda": torch.version.cuda,
               "gpu": torch.cuda.get_device_name(0), "cudnn": torch.backends.cudnn.version()}
print(environment)

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def save_csv(frame, path):
    temporary = path.with_suffix(".tmp.csv")
    frame.to_csv(temporary, index=False)
    os.replace(temporary, path)

## 2. Original group data and unchanged split

The hash must match the **306 local images inspected in your project**. A different or
modified dataset stops the notebook instead of silently substituting data. Original split:
244 training images (68/75/41/60) and 62 validation images (17/19/11/15), in class order
apple/cherry/chestnut/maple. All images are already 96×96 RGB, so B0 uses exactly the
Task 3 `ToTensor` and `Normalize` preprocessing, without an additional resize.

The folders are never reshuffled into new splits. No validation image is used for training.
No exact file/RGB duplicates were found locally. Photographer-level independence cannot
be established from the available metadata alone; preserve the teacher's supplied split.

In [ ]:
GROUP_ZIP = GROUP_ZIP.strip()
link = urlparse(GROUP_ZIP)
assert link.hostname == "drive.google.com" and "/folders/" not in link.path, "Provide the teacher's Google Drive ZIP file URL."
file_id = link.path.split("/d/", 1)[1].split("/")[0] if "/d/" in link.path else parse_qs(link.query).get("id", [""])[0]
assert file_id, "Use a Google Drive file link containing /d/FILE_ID or ?id=FILE_ID."
data_dir = Path("/content") / ("ex08_group_" + hashlib.sha256(GROUP_ZIP.encode()).hexdigest()[:12])
DATA = data_dir / "leaves"
if not DATA.exists():
    data_dir.mkdir(parents=True, exist_ok=True)
    archive = data_dir / "group.zip"
    downloaded = gdown.download(id=file_id, output=str(archive), quiet=True)
    assert downloaded and zipfile.is_zipfile(archive), "The link must download the original group ZIP."
    with zipfile.ZipFile(archive) as z:
        assert all(any(n.startswith("leaves/" + split + "/") for n in z.namelist()) for split in ["train", "val"])
        z.extractall(data_dir)

classes = ["apple", "cherry", "chestnut", "maple"]
records = []
counts = {}
for split in ["train", "val"]:
    assert sorted(p.name for p in (DATA / split).iterdir() if p.is_dir()) == classes
    for name in classes:
        paths = sorted((DATA / split / name).glob("*"))
        counts[split + "/" + name] = len(paths)
        for p in paths:
            with Image.open(p) as image:
                image.load()
                assert image.size == (96, 96) and image.mode == "RGB", str(p.relative_to(DATA))
            records.append({"path": p.relative_to(DATA).as_posix(), "sha256": hashlib.sha256(p.read_bytes()).hexdigest()})
records.sort(key=lambda row: row["path"])
data_sha = hashlib.sha256("".join(r["path"] + "\0" + r["sha256"] + "\n" for r in records).encode()).hexdigest()
assert data_sha == EXPECTED_DATASET_SHA256, "This ZIP differs from the inspected original group images. Check the URL."
assert len({r["sha256"] for r in records}) == len(records), "Duplicate image files detected."
print(counts)
print("Validation: 62 images; one correct prediction =", 100 / 62, "percentage points.")
mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize(mean, std)])

## 3. Task 3 dataset and network

`LeafDataset` is copied unchanged. `LeafNet` retains its four convolutions, four max pools
and two linear layers. The only additions are the optional dropout and global-average-pooling
switches needed for E2 and E4. B0 uses `dropout=0`, `gap=False`: the original **208,292**
parameters and forward computation. Dropout is after the hidden ReLU; GAP averages the
last feature map and changes the input width of `fc1` from 2304 to 64.

B0 reproduces Tasks **3b–3c**, not the later continued-training cell. That cell defined
`LeafNetBN` but never instantiated it; it continued training the previous `LeafNet` for
15 extra epochs with augmentation. This notebook does not silently turn that into a new baseline.

In [ ]:
class LeafDataset(Dataset):
    def __init__(self, folder, transform):
        self.transform = transform
        folder_path = Path(folder)
        # The class names: the sorted subfolders of the folder.
        self.classes = sorted([d.name for d in folder_path.iterdir() if d.is_dir()])

        # Map class name to class index
        class_to_idx = {cls_name: i for i, cls_name in enumerate(self.classes)}

        # A list of (file path, class index) pairs: every class, its files in sorted order.
        self.samples = []
        for cls_name in self.classes:
            cls_dir = folder_path / cls_name
            # Gather and sort all files in this directory
            file_paths = sorted(list(cls_dir.glob("*")))
            for path in file_paths:
                if path.is_file():
                    self.samples.append((str(path), class_to_idx[cls_name]))

    def __len__(self):
        # The number of samples.
        return len(self.samples)

    def __getitem__(self, i):
        # Open the i-th image as RGB, apply the transform, return it with its class index.
        img_path, class_idx = self.samples[i]
        img = Image.open(img_path).convert("RGB")
        if self.transform is not None:
            img = self.transform(img)
        return img, class_idx


class LeafNet(nn.Module):
    def __init__(self, dropout=0.0, gap=False):
        super().__init__()
        # The layers as attributes: the four convolutions, one max pooling, the two linear layers.
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.conv4 = nn.Conv2d(64, 64, kernel_size=3, padding=1)

        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        # After 4 pools, a 96x96 image becomes 6x6. The last conv output has 64 channels.
        self.gap = gap
        self.fc1 = nn.Linear(64 if gap else 64 * 6 * 6, 64)
        self.fc2 = nn.Linear(64, 4)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # Every block: convolution, ReLU, pooling. Then flatten and the two linear layers.
        x = self.pool(torch.relu(self.conv1(x)))
        x = self.pool(torch.relu(self.conv2(x)))
        x = self.pool(torch.relu(self.conv3(x)))
        x = self.pool(torch.relu(self.conv4(x)))

        x = x.mean(dim=(2, 3)) if self.gap else x.flatten(1)
        x = torch.relu(self.fc1(x))
        return self.fc2(self.dropout(x))

train_ds = LeafDataset(DATA / "train", transform)
val_ds = LeafDataset(DATA / "val", transform)
assert train_ds.classes == val_ds.classes == classes
assert (len(train_ds), len(val_ds)) == (244, 62)
assert sum(p.numel() for p in LeafNet().parameters()) == 208292

## 4. Generate the camera-like validation set once

One fixed distorted image is saved for each of the 62 validation images: random resized
crop (65–100% area, aspect ratio 0.9–1.1), rotation ±20°, and brightness/contrast/saturation
±30%, hue ±0.04. Each image uses seed `CAMERA_SEED + index`. PNGs and their hashes remain
in private Drive storage and are reused unchanged by every model and training seed.
The camera set is a controlled robustness check, not a separate real-camera test set.

In [ ]:
camera_transform = transforms.Compose([
    transforms.RandomResizedCrop(96, scale=(0.65, 1.0), ratio=(0.9, 1.1)),
    transforms.RandomRotation(20, interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.04)
])
camera_spec = {"dataset": data_sha, "seed": CAMERA_SEED, "transform": str(camera_transform),
               "torch": torch.__version__, "torchvision": torchvision.__version__, "pillow": PIL.__version__}
camera_key = hashlib.sha256(json.dumps(camera_spec, sort_keys=True).encode()).hexdigest()[:12]
camera_dir = ROOT / ("camera_val_" + camera_key)
manifest_path = camera_dir / "manifest.json"
if not manifest_path.exists():
    camera_records = []
    for i, (path, label) in enumerate(val_ds.samples):
        seed_everything(CAMERA_SEED + i)
        target = camera_dir / classes[label] / (Path(path).stem + ".png")
        target.parent.mkdir(parents=True, exist_ok=True)
        with Image.open(path) as image:
            camera_transform(image.convert("RGB")).save(target)
        camera_records.append({"path": target.relative_to(camera_dir).as_posix(),
                               "sha256": hashlib.sha256(target.read_bytes()).hexdigest()})
    manifest_path.write_text(json.dumps({"spec": camera_spec, "files": camera_records}, indent=2))
camera_manifest = json.loads(manifest_path.read_text())
assert camera_manifest["spec"] == camera_spec
for row in camera_manifest["files"]:
    assert hashlib.sha256((camera_dir / row["path"]).read_bytes()).hexdigest() == row["sha256"]
camera_ds = LeafDataset(camera_dir, transform)
assert camera_ds.classes == classes and len(camera_ds) == len(val_ds) == 62
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
camera_loader = DataLoader(camera_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
print("Fixed camera-like validation ready:", len(camera_ds), "images; manifest", camera_key)

## 5. Variant settings — one change against the stated baseline

| Variant | Comparison | Change |
|---|---|---|
| B0 | Task 3b–3c | Original scratch CNN, Adam, LR 0.001, 15 epochs |
| E1 | B0 | ColorJitter only (same colour ranges as above) |
| E2 | B0 | Hidden dropout 0.3 |
| E3 | B0 | AdamW with weight decay 0.01 (an optimizer/decay package) |
| E4 | B0 | Global average pooling instead of spatial flattening |
| E5 | E6 | Pretrained MobileNetV3 Small; only its new last linear layer trained |
| E6 | B0 as a reference | Same pretrained MobileNet; full-network fine-tuning |
| E7 | E6 | Colour plus geometric augmentation as a package |
| E8 | E6 | Cosine scheduler, minimum LR 0.00001 |
| E9 | B0 or E6 | Conditional combination of supported isolated changes |

All runs use the same 244 training images, batch size 32, 15 epochs, seeds and 96×96
ImageNet-normalized inputs. E5/E6 use the same ImageNet1K V1 weights and new-layer
initialization for a given seed. E5 freezes feature batch-normalization statistics as well
as the weights; its classifier dropout remains active. MobileNet's existing dropout 0.2 is
retained in every transfer variant. B0 vs E6 changes architecture **and** initialization;
it does not isolate the effect of pretraining. E7 does not isolate colour from geometry.

Final-epoch weights are compared throughout; there is no extra best-epoch search.
Per-epoch seeding is added for reproducible restarts, with a separate DataLoader generator
so augmentation and dropout cannot change the training sample order. The original
Task 3 mean-of-batch training loss is retained (the final batch has 20 images).

In [ ]:
base = dict(model="scratch", color=False, geometry=False, dropout=0.0, gap=False,
            optimizer="Adam", weight_decay=0.0, frozen=False, cosine=False, baseline="B0")
variants = {
    "B0": {}, "E1": dict(color=True), "E2": dict(dropout=0.3),
    "E3": dict(optimizer="AdamW", weight_decay=0.01), "E4": dict(gap=True),
    "E5": dict(model="mobile", frozen=True, baseline="E6"),
    "E6": dict(model="mobile"),
    "E7": dict(model="mobile", color=True, geometry=True, baseline="E6"),
    "E8": dict(model="mobile", cosine=True, baseline="E6")
}
spec = dict(version=1, environment=environment, dataset_sha256=data_sha,
            camera=camera_manifest, seeds=SEEDS, epochs=EPOCHS, batch_size=BATCH_SIZE,
            lr=LR, base=base, variants=variants, selection="final epoch; camera accuracy first")
run_key = hashlib.sha256(json.dumps(spec, sort_keys=True).encode()).hexdigest()[:12]
OUT = ROOT / ("runs_" + run_key)
OUT.mkdir(exist_ok=True)
(OUT / "configuration.json").write_text(json.dumps(spec, indent=2))
(OUT / "dataset_manifest.json").write_text(json.dumps(records, indent=2))
results_path = OUT / "results.csv"
results = pd.read_csv(results_path).to_dict("records") if results_path.exists() else []
print("Persistent run folder:", OUT)

def make_model(cfg):
    if cfg["model"] == "scratch":
        return LeafNet(dropout=cfg["dropout"], gap=cfg["gap"])
    model = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.IMAGENET1K_V1)
    model.classifier[3] = nn.Linear(model.classifier[3].in_features, 4)
    if cfg["frozen"]:
        for p in model.parameters():
            p.requires_grad = False
        for p in model.classifier[3].parameters():
            p.requires_grad = True
    return model

def evaluate(model, loader):
    model.eval()
    correct = 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            correct += (model(images).argmax(dim=1) == labels).sum().item()
    return correct

## 6. Task 3 training loop with per-epoch checkpoints and measurements

Each completed epoch saves model/optimizer/scheduler state and history to Drive before
continuing. CSV files are replaced via temporary files. Evaluation uses only `eval()` and
`no_grad()`. Training seconds measure the training batches (including data loading), with
CUDA synchronized at the boundaries; validation and weight-download time are excluded.
Only fully trained runs enter `results.csv`; exceptions are written to `failures.log`.

In [ ]:
def run_experiment(name, seed, cfg):
    folder = OUT / f"{name}_seed{seed}"
    folder.mkdir(exist_ok=True)
    config_path = folder / "config.json"
    run_config = dict(variant=name, seed=seed, settings=cfg, epochs=EPOCHS, lr=LR, batch_size=BATCH_SIZE)
    if config_path.exists():
        assert json.loads(config_path.read_text()) == run_config, "Refusing to mix different run configurations."
    config_path.write_text(json.dumps(run_config, indent=2))
    seed_everything(seed)
    model = make_model(cfg).to(device)
    augment = []
    if cfg["geometry"]:
        augment += [transforms.RandomResizedCrop(96, scale=(0.6, 1.0)),
                    transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip(), transforms.RandomRotation(20)]
    if cfg["color"]:
        augment += [transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.04)]
    train_transform = transforms.Compose(augment + [transforms.ToTensor(), transforms.Normalize(mean, std)])
    my_train_ds = LeafDataset(DATA / "train", train_transform)
    order = torch.Generator()
    my_train_loader = DataLoader(my_train_ds, batch_size=BATCH_SIZE, shuffle=True, generator=order, num_workers=0)
    criterion = nn.CrossEntropyLoss()
    optimizer_class = torch.optim.AdamW if cfg["optimizer"] == "AdamW" else torch.optim.Adam
    optimizer = optimizer_class((p for p in model.parameters() if p.requires_grad), lr=LR, weight_decay=cfg["weight_decay"])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5) if cfg["cosine"] else None
    checkpoint = folder / "checkpoint.pt"
    history = []
    if checkpoint.exists():
        saved = torch.load(checkpoint, map_location=device, weights_only=True)
        model.load_state_dict(saved["model"])
        optimizer.load_state_dict(saved["optimizer"])
        if scheduler:
            scheduler.load_state_dict(saved["scheduler"])
        history = saved["history"]
        save_csv(pd.DataFrame(history), folder / "history.csv")
        print(name, seed, "resuming after epoch", len(history))
    for epoch in range(len(history), EPOCHS):
        epoch_seed = seed + 1000 * epoch
        seed_everything(epoch_seed)
        order.manual_seed(epoch_seed)
        model.train()
        if cfg["frozen"]:
            model.features.eval()
        loss_sum = 0.0
        lr = optimizer.param_groups[0]["lr"]
        torch.cuda.synchronize()
        start = time.perf_counter()
        for images, labels in my_train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = criterion(model(images), labels)
            loss.backward()
            optimizer.step()
            loss_sum += loss.item()
        torch.cuda.synchronize()
        seconds = time.perf_counter() - start
        correct = evaluate(model, val_loader)
        camera_correct = evaluate(model, camera_loader)
        history.append(dict(epoch=epoch + 1, training_loss=loss_sum / len(my_train_loader),
                            val_correct=correct, val_total=len(val_ds), val_accuracy=correct / len(val_ds),
                            camera_correct=camera_correct, camera_total=len(camera_ds), camera_accuracy=camera_correct / len(camera_ds),
                            epoch_train_seconds=seconds, train_seconds=seconds + (history[-1]["train_seconds"] if history else 0.0), lr=lr))
        if scheduler:
            scheduler.step()
        temporary = folder / "checkpoint.tmp"
        torch.save(dict(model=model.state_dict(), optimizer=optimizer.state_dict(),
                        scheduler=scheduler.state_dict() if scheduler else None, history=history), temporary)
        os.replace(temporary, checkpoint)
        save_csv(pd.DataFrame(history), folder / "history.csv")
        print(f"{name} seed {seed} epoch {epoch + 1:2d}: loss {history[-1]['training_loss']:.4f}, val {correct}/62, camera {camera_correct}/62, train {seconds:.1f}s")
    # Also reconstruct the result if a disconnect happened after the last checkpoint.
    row = dict(variant=name, seed=seed, status="completed", epochs=EPOCHS,
               parameters=sum(p.numel() for p in model.parameters()),
               trainable_parameters=sum(p.numel() for p in model.parameters() if p.requires_grad),
               batch_size=BATCH_SIZE, initial_lr=LR, **cfg, **history[-1])
    results[:] = [r for r in results if (r["variant"], r["seed"]) != (name, seed)] + [row]
    save_csv(pd.DataFrame(results), results_path)
    del model, optimizer
    torch.cuda.empty_cache()

def execute(plan):
    for seed in SEEDS:
        for name in plan:
            if any(r["variant"] == name and r["seed"] == seed for r in results):
                print(name, seed, "already completed")
                continue
            try:
                run_experiment(name, seed, base | variants[name])
            except Exception:
                message = f"FAILED {name}, seed {seed}\n" + traceback.format_exc()
                print(message)
                with (OUT / "failures.log").open("a") as log:
                    log.write(message + "\n")
                torch.cuda.empty_cache()

execute(["B0", "E1", "E5", "E6", "E7"])
if RUN_ALL:
    execute(["E2", "E3", "E4", "E8"])

## 7. Conditional combination E9

An isolated change is considered promising here only after all three matched seeds finish:
its camera accuracy must never decrease, its mean gain must be at least one validation image,
and its mean gain must exceed the larger of the two across-seed standard deviations.
This is a conservative descriptive rule, **not a significance test**.

Combine supported scratch changes if at least two qualify; otherwise combine E7 and E8
if both qualify against E6. A single supported change is not called a combination. If both
families qualify, prefer the family whose best isolated mean camera score is higher.
The first selected E9 configuration is saved and reused, so later restarts cannot change it.
If evidence is insufficient, E9 is explicitly skipped. Judge these choices yourself for your report.

In [ ]:
frame = pd.DataFrame(results)
def supported(name, baseline):
    if frame.empty:
        return False
    a = frame[frame.variant == name].set_index("seed").reindex(SEEDS).camera_accuracy
    b = frame[frame.variant == baseline].set_index("seed").reindex(SEEDS).camera_accuracy
    if a.isna().any() or b.isna().any():
        return False
    gain = a - b
    return bool((gain >= -1e-12).all() and gain.mean() >= 1 / 62 - 1e-12 and gain.mean() > max(a.std(), b.std()))

e9_path = OUT / "E9_configuration.json"
if e9_path.exists():
    variants["E9"] = json.loads(e9_path.read_text())
else:
    combinations = []
    good_scratch = [name for name in ["E1", "E2", "E3", "E4"] if supported(name, "B0")]
    if len(good_scratch) >= 2:
        combined = {key: value for name in good_scratch for key, value in variants[name].items()}
        combinations.append((frame[frame.variant.isin(good_scratch)].groupby("variant").camera_accuracy.mean().max(), combined))
    if supported("E7", "E6") and supported("E8", "E6"):
        combinations.append((frame[frame.variant.isin(["E7", "E8"])].groupby("variant").camera_accuracy.mean().max(), variants["E7"] | dict(cosine=True)))
    if combinations:
        variants["E9"] = max(combinations, key=lambda item: item[0])[1]
        e9_path.write_text(json.dumps(variants["E9"], indent=2))
if "E9" in variants:
    print("E9 settings:", variants["E9"])
    execute(["E9"])
else:
    print("E9 NOT RUN: insufficient repeated isolated evidence for a combination.")

## 8. Actual results, variability and comparison chart

Accuracies are fractions; multiply by 100 for percent. `*_std` is the sample standard
deviation across training seeds, not uncertainty over new photographers. Each original
and distorted validation image contributes 1/62 = 1.61 percentage points.
Camera-like images come from the same validation set, so model selection remains a
development-set comparison, with no independent real-phone test.

The table of deltas compares each variant with its declared baseline on matched seeds.
It helps identify changes that helped or failed and whether gains are larger than observed
seed variation. A missing or failed experiment never receives an invented accuracy.

In [ ]:
frame = pd.DataFrame(results)
assert not frame.empty, "No training run completed. Read failures.log and rerun after fixing the cause."
summary = frame.groupby("variant").agg(n=("seed", "count"), val_mean=("val_accuracy", "mean"),
    val_std=("val_accuracy", "std"), camera_mean=("camera_accuracy", "mean"),
    camera_std=("camera_accuracy", "std"), training_loss_mean=("training_loss", "mean"),
    train_seconds_mean=("train_seconds", "mean"), parameters=("parameters", "first")).reset_index()
save_csv(summary, OUT / "summary.csv")
display(summary.round(4))
deltas = []
for name in frame.variant.unique():
    baseline = (base | variants[name])["baseline"]
    if name == baseline:
        continue
    paired = frame[frame.variant == name].merge(frame[frame.variant == baseline], on="seed", suffixes=("", "_baseline"))
    if len(paired):
        dv = paired.val_accuracy - paired.val_accuracy_baseline
        dc = paired.camera_accuracy - paired.camera_accuracy_baseline
        deltas.append(dict(variant=name, baseline=baseline, paired_seeds=len(paired),
                           val_gain_mean=dv.mean(), val_gain_std=dv.std(), camera_gain_mean=dc.mean(), camera_gain_std=dc.std()))
save_csv(pd.DataFrame(deltas), OUT / "paired_comparisons.csv")
display(pd.DataFrame(deltas).round(4))
fig, ax = plt.subplots(figsize=(10, 4))
x = np.arange(len(summary))
ax.bar(x - 0.18, summary.val_mean * 100, 0.36, yerr=summary.val_std.fillna(0) * 100, label="Original validation")
ax.bar(x + 0.18, summary.camera_mean * 100, 0.36, yerr=summary.camera_std.fillna(0) * 100, label="Camera-like validation")
ax.set(xticks=x, xticklabels=summary.variant, ylabel="Accuracy (%)", ylim=(0, 105))
ax.legend()
fig.tight_layout()
fig.savefig(OUT / "comparison.png", dpi=160)
plt.show()
statuses = []
for name in ["B0", "E1", "E2", "E3", "E4", "E5", "E6", "E7", "E8", "E9"]:
    for seed in SEEDS:
        done = any(r["variant"] == name and r["seed"] == seed for r in results)
        failed = (OUT / f"{name}_seed{seed}" / "config.json").exists()
        statuses.append(dict(variant=name, seed=seed, status="completed" if done else "failed_or_interrupted" if failed else "not_run"))
save_csv(pd.DataFrame(statuses), OUT / "experiment_status.csv")
print("CSV files, checkpoints and per-epoch histories:", OUT)

## 9. Separate candidate model and ONNX verification

Choose the variant with the highest **mean camera-like accuracy across completed seeds**;
break ties with original validation accuracy, then fewer parameters (a size/speed proxy).
Prefer variants with all three seeds complete. Within that variant, save the seed with
the best camera score, breaking ties with original accuracy and then seed number.
This validation-selected candidate still needs your real-phone evaluation.

Export uses the existing browser contract: central-square crop outside the model,
96×96 RGB, divide by 255, ImageNet mean/std, float32 NCHW `[1,3,96,96]`, raw logits.
The supplied validation images already have the crop and resize applied.
The ONNX file embeds `model_info` for the course camera application and uses opset 17.
Compare PyTorch and ONNX probabilities and predictions on **all 124** original/distorted
validation inputs. Measure ONNX CPU inference time on the Colab host; this is **not phone speed**.

The download ZIP contains the candidate ONNX, preprocessing metadata and result tables.
It excludes original photographs, dataset URLs and training checkpoints. The checkpoint
and full logs stay on private Drive. Nothing is uploaded to GitHub or the existing website.

In [ ]:
import onnx
import onnxruntime as ort

eligible = summary[summary.n == len(SEEDS)]
if eligible.empty:
    print("WARNING: no variant has all requested seeds; candidate selection uses incomplete repeats.")
    eligible = summary
winning_variant = eligible.sort_values(["camera_mean", "val_mean", "parameters", "variant"], ascending=[False, False, True, True]).iloc[0].variant
best = frame[frame.variant == winning_variant].sort_values(["camera_accuracy", "val_accuracy", "seed"], ascending=[False, False, True]).iloc[0]
cfg = base | variants[winning_variant]
candidate = make_model(cfg)
saved = torch.load(OUT / f"{winning_variant}_seed{int(best.seed)}" / "checkpoint.pt", map_location="cpu", weights_only=True)
candidate.load_state_dict(saved["model"])
candidate.eval()
candidate_dir = OUT / "candidate"
candidate_dir.mkdir(exist_ok=True)
torch.save(candidate.state_dict(), candidate_dir / "candidate_state.pt")
model_info = dict(classes=classes, input_size=96, mean=mean, std=std)
(candidate_dir / "model_info.json").write_text(json.dumps(model_info, indent=2))
onnx_path = candidate_dir / "candidate_leaf_model.onnx"
torch.onnx.export(candidate, (torch.zeros(1, 3, 96, 96),), str(onnx_path),
                  input_names=["image"], output_names=["logits"], opset_version=17,
                  dynamo=False, external_data=False)
exported = onnx.load(str(onnx_path))
exported.metadata_props.add(key="model_info", value=json.dumps(model_info))
onnx.checker.check_model(exported)
onnx.save(exported, str(onnx_path))
options = ort.SessionOptions()
options.intra_op_num_threads = 1
options.inter_op_num_threads = 1
session = ort.InferenceSession(str(onnx_path), sess_options=options, providers=["CPUExecutionProvider"])
max_error, agreement, total = 0.0, 0, 0
with torch.no_grad():
    for dataset in [val_ds, camera_ds]:
        for i in range(len(dataset)):
            image, _ = dataset[i]
            x = image.unsqueeze(0)
            expected = candidate(x).softmax(1).numpy()
            logits = session.run(None, {"image": x.numpy()})[0]
            p = np.exp(logits - logits.max(1, keepdims=True))
            p /= p.sum(1, keepdims=True)
            np.testing.assert_allclose(p, expected, rtol=1e-3, atol=1e-5)
            max_error = max(max_error, float(np.max(np.abs(p - expected))))
            agreement += int(p.argmax() == expected.argmax())
            total += 1
assert agreement == total == 124
for _ in range(5):
    session.run(None, {"image": x.numpy()})
latencies = []
for _ in range(30):
    start = time.perf_counter()
    session.run(None, {"image": x.numpy()})
    latencies.append(1000 * (time.perf_counter() - start))
verification = dict(variant=winning_variant, seed=int(best.seed), validation_inputs=total,
    matching_predictions=agreement, max_probability_error=max_error,
    onnx_bytes=onnx_path.stat().st_size, colab_cpu_median_ms=float(np.median(latencies)),
    colab_cpu_p95_ms=float(np.percentile(latencies, 95)), phone_speed_tested=False)
(candidate_dir / "verification.json").write_text(json.dumps(verification, indent=2))
print(verification)
bundle = OUT / "candidate_for_camera.zip"
with zipfile.ZipFile(bundle, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for p in [onnx_path, candidate_dir / "model_info.json", candidate_dir / "verification.json",
              results_path, OUT / "summary.csv", OUT / "paired_comparisons.csv", OUT / "experiment_status.csv", OUT / "comparison.png"]:
        z.write(p, p.name)
print("Candidate saved separately; existing deployed model was not changed:", candidate_dir)
files.download(str(bundle))

## What still requires your evaluation

Inspect `results.csv`, `summary.csv`, `paired_comparisons.csv`, `experiment_status.csv`,
`comparison.png` and the per-run `history.csv` files. Compare gains in units of correct
images and against the seed variation. Failed and unrun variants have no accuracy result.
Use the candidate ONNX in the course camera application and assess new leaves, lighting,
backgrounds and latency on your phone. Synthetic distortions and Colab CPU timing cannot
establish real-camera performance. Decide whether the candidate deserves deployment;
this notebook does not deploy it. Write your own `REPORT.md` and `REVIEW.md`.

**Local verification before delivery:** notebook schema and Python syntax, exact reuse of
`LeafDataset`, the limited `LeafNet` changes, dataset hash/counts/format/duplicates, and
preservation of the original notebook, app and ONNX files. **Not executed locally:**
Colab installation/authentication/downloads, CUDA training, interruption/resume on Drive,
candidate selection from actual runs, ONNX export/parity and inference timing. These
checks execute above in Colab; this unexecuted notebook contains no experiment claims.